# Notebook para treinamento e testes de modelos

In [1]:
import pandas as pd
import numpy as np

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
import prophet

from sklearn.model_selection import KFold
import matplotlib.pyplot as plt
import datetime
from dateutil.relativedelta import relativedelta
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
optuna.logging.set_verbosity(optuna.logging.ERROR)

c:\Users\jhiy2\Documents\trabalho_icd\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


In [2]:
horizonte_previsao = 3
tamanho_teste = 6
n_trials = 3
metrica_erro = 'mae'
tolerancia_fipe = 200
tolerancia_exog = 0.01

skus_teste = [0, 1, 2, 3, 4]

## Leitura da base de dados

In [3]:
# Base fipe historica
fipe_path = './data/dados_fipe_tratados.csv'
# Base da taxa de cambio
exchange_path = './data/DEXBZUS_tratados.csv'
# Base IPCA
ipca_path = './data/bcdata.sgs.433_tratados.csv' 

df_fipe = pd.read_csv(fipe_path)
print('FIPE shape:', df_fipe.shape)
print(df_fipe.head())

df_ex = pd.read_csv(exchange_path)
print('DEXBZUS head:')
print(df_ex.head())

df_ipca = pd.read_csv(ipca_path)
print('IPCA head:')
print(df_ipca.head())

FIPE shape: (411498, 9)
   Unnamed: 0       reference_date brand_name          model_name  year  \
0           0  2021-01-01 00:00:00       Fiat           147 C/ CL  1987   
1           1  2021-01-01 00:00:00       Fiat           147 C/ CL  1986   
2           2  2021-01-01 00:00:00       Fiat           147 C/ CL  1985   
3           3  2021-01-01 00:00:00       Fiat  147 Furgão (todos)  1987   
4           4  2021-01-01 00:00:00       Fiat  147 Furgão (todos)  1986   

  fuel_name  brl_price  year_of_reference month_of_reference  
0  Gasolina     2723.0               2021            January  
1  Gasolina     2484.0               2021            January  
2  Gasolina     2324.0               2021            January  
3  Gasolina     2199.0               2021            January  
4  Gasolina     2094.0               2021            January  
DEXBZUS head:
         date  exchange_rate
0  1995-01-01       0.846091
1  1995-02-01       0.841150
2  1995-03-01       0.890522
3  1995-04-01    

In [4]:
df_ipca['date'] = pd.to_datetime(df_ipca['date'])
df_ex['date'] = pd.to_datetime(df_ex['date'])

df_ipca.index = df_ipca['date']
df_ex.index = df_ex['date']

In [5]:
df_fipe = df_fipe.drop(columns=['Unnamed: 0'])

In [6]:
df_fipe['reference_date'] = pd.to_datetime(df_fipe['reference_date'], format='ISO8601')

df_fipe['sku'] = df_fipe.groupby(['brand_name', 'model_name', 'fuel_name', 'year']).ngroup()

In [7]:
# Retirar isso depois
df_fipe = df_fipe[df_fipe['reference_date'].dt.year > 2022]

In [8]:
df_fipe.head()

,reference_date,brand_name,model_name,year,fuel_name,brl_price,year_of_reference,month_of_reference,sku
125101,2023-09-01,Fiat,147 C/ CL,1987,Gasolina,4630.0,2023,September,2
125102,2023-09-01,Fiat,147 C/ CL,1986,Gasolina,4478.0,2023,September,1
125103,2023-09-01,Fiat,147 C/ CL,1985,Gasolina,3898.0,2023,September,0
125104,2023-09-01,Fiat,147 Furgão (todos),1987,Gasolina,2637.0,2023,September,5
125105,2023-09-01,Fiat,147 Furgão (todos),1986,Gasolina,2528.0,2023,September,4


In [9]:
df_fipe = df_fipe.drop(columns=['year_of_reference', 'month_of_reference'])

In [10]:
df_fipe.columns

Index(['reference_date', 'brand_name', 'model_name', 'year', 'fuel_name',
       'brl_price', 'sku'],
      dtype='str')

In [11]:
df_previsao_list = []

data_ref = pd.to_datetime('2026-04-01')

for sku in skus_teste:
    df = df_fipe.query("sku == @sku").copy()
    df = df.set_index('reference_date')

    meses_totais = relativedelta(data_ref, df.index.min()).years * 12 + relativedelta(data_ref, df.index.min()).months

    if meses_totais < 12:
        print(f"SKU: {sku} não tem dados suficientes ({meses_totais} meses apenas)")
        continue

    meses_esperados = pd.date_range(f'{df.index.min().year}-{df.index.min().month}', f'{data_ref.year}-{data_ref.month}', freq='MS')

    df = df.reindex(meses_esperados)

    df['brand_name'] = df['brand_name'].ffill()
    df['model_name'] = df['model_name'].ffill()
    df['year'] = df['year'].ffill()
    df['fuel_name'] = df['fuel_name'].ffill()

    df['brl_price'] = df['brl_price'].interpolate(method='linear')

    df = df.rename_axis('reference_date').reset_index()

    df.index = df['reference_date']
    df['sku'] = df['sku'].ffill()

    df_previsao_list.append(df)

df_previsao = pd.concat(df_previsao_list, ignore_index=True)

## Separar os dados em treino e teste para exógenas

In [12]:
data_ref = pd.to_datetime('2026-04-01')
start = data_ref + relativedelta(months=-tamanho_teste)

train_ipca = df_ipca[df_ipca.index <= start]

test_ipca = df_ipca[df_ipca.index > start]

train_ex = df_ex[df_ex.index <= start]

test_ex = df_ex[df_ex.index > start]

exog_train = pd.concat([train_ipca['valor'], train_ex['exchange_rate']], axis=1)
exog_test = pd.concat([test_ipca['valor'], test_ex['exchange_rate']], axis=1)


df_prophet = pd.concat([df, df_ipca, df_ex], axis=1, sort=False)
df_prophet = df_prophet[['reference_date', 'date', 'brl_price', 'valor', 'exchange_rate']]
df_prophet = df_prophet.rename(columns={
    'reference_date': 'ds',
    'brl_price': 'y'
})

df_train_prophet = df_prophet[
    df_prophet['ds'] <= start
]
df_test_prophet = df_prophet[
    df_prophet['ds'] > start
]

df_train_prophet = df_train_prophet.reset_index(drop=True)
df_test_prophet = df_test_prophet.reset_index(drop=True)

## Geradores de modelos SARIMAX e ETS

In [13]:
from model_generators import generate_ets_model, generate_sarimax_model, generate_prophet_model, feature_engineering, criar_ets_fipe_real, criar_sarimax_fipe_real, criar_prophet_fipe_real

## Modelos

#### Modelo do Câmbio

##### Prophet

In [14]:
df_train_exchange = df_train_prophet[['ds', 'exchange_rate']].rename(columns={'exchange_rate': 'y'})
df_test_exchange = df_test_prophet[['ds', 'exchange_rate']].rename(columns={'exchange_rate': 'y'})

model_exchange, info_exchange_prophet, best_value_exchange_prophet = generate_prophet_model(
    df_train_exchange,
    df_test_exchange,
    [],
    n_trials,
    metrica_erro,
    tolerancia_exog
)

model_exchange.fit(
    df_train_exchange,
)

forecast_exchange = model_exchange.predict(
    df_test_exchange[['ds']]
)

pd.concat([df_test_exchange['y'], forecast_exchange['yhat']], axis=1)


  0%|          | 0/3 [00:00<?, ?it/s]16:27:19 - cmdstanpy - INFO - Chain [1] start processing
16:27:19 - cmdstanpy - INFO - Chain [1] done processing
Best trial: 0. Best value: 0.866433:  33%|███▎      | 1/3 [00:00<00:00,  3.63it/s]16:27:19 - cmdstanpy - INFO - Chain [1] start processing
16:27:19 - cmdstanpy - INFO - Chain [1] done processing
Best trial: 1. Best value: 0.845471:  67%|██████▋   | 2/3 [00:00<00:00,  4.05it/s]16:27:19 - cmdstanpy - INFO - Chain [1] start processing
16:27:19 - cmdstanpy - INFO - Chain [1] done processing
Best trial: 2. Best value: 0.780723: 100%|██████████| 3/3 [00:00<00:00,  4.50it/s]
16:27:20 - cmdstanpy - INFO - Chain [1] start processing
16:27:20 - cmdstanpy - INFO - Chain [1] done processing


,y,yhat
0,5.341483,5.860841
1,5.455709,6.010461
2,5.331620,6.052331
3,5.198805,5.966658
4,5.229641,5.985032
5,5.033945,5.993249


##### SARIMAX

In [15]:
model, info_exchange_sarimax, best_value_exchange_sarimax = generate_sarimax_model(train_ex['exchange_rate'], test_ex['exchange_rate'], None, None, n_trials, metrica_erro, tolerancia_exog)

results = model.fit(disp=False)

forecasts_ex_sarimax = results.forecast(steps=len(test_ex['exchange_rate']))

  0%|          | 0/3 [00:00<?, ?it/s]c:\Users\jhiy2\Documents\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\jhiy2\Documents\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
Best trial: 0. Best value: 0.193733:  33%|███▎      | 1/3 [00:03<00:06,  3.03s/it]c:\Users\jhiy2\Documents\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\jhiy2\Documents\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_date

In [16]:
print(f"Valor da métrica: {best_value_exchange_sarimax}")
display(pd.concat([forecasts_ex_sarimax, test_ex['exchange_rate']], axis=1))

Valor da métrica: 0.09896190129782066


,predicted_mean,exchange_rate
2025-11-01,5.389675,5.341483
2025-12-01,5.412276,5.455709
2026-01-01,5.396620,5.331620
2026-02-01,5.389675,5.198805
2026-03-01,5.417188,5.229641
2026-04-01,5.398121,5.033945


#### Modelo do IPCA

##### Prophet

In [17]:
df_train_ipca = df_train_prophet[['ds', 'valor']].rename(columns={'valor': 'y'})
df_test_ipca = df_test_prophet[['ds', 'valor']].rename(columns={'valor': 'y'})

df_train_prophet = df_train_prophet.ffill()
df_test_prophet = df_test_prophet.ffill()

model_ipca, info_ipca_prophet, best_value_ipca_prophet = generate_prophet_model(
    df_train_ipca,
    df_test_ipca,
    [],
    n_trials,
    metrica_erro,
    tolerancia_exog
)

model_ipca.fit(
    df_train_ipca
)

forecast_ipca = model_ipca.predict(
    df_test_ipca[['ds']]
)

pd.concat([df_test_ipca['y'], forecast_ipca['yhat']], axis=1)

  0%|          | 0/3 [00:00<?, ?it/s]16:27:28 - cmdstanpy - INFO - Chain [1] start processing
16:27:28 - cmdstanpy - INFO - Chain [1] done processing
Best trial: 0. Best value: 0.230844:  33%|███▎      | 1/3 [00:00<00:00,  4.05it/s]16:27:28 - cmdstanpy - INFO - Chain [1] start processing
16:27:28 - cmdstanpy - INFO - Chain [1] done processing
Best trial: 1. Best value: 0.224732:  67%|██████▋   | 2/3 [00:00<00:00,  4.61it/s]16:27:28 - cmdstanpy - INFO - Chain [1] start processing
16:27:28 - cmdstanpy - INFO - Chain [1] done processing
Best trial: 2. Best value: 0.224504: 100%|██████████| 3/3 [00:00<00:00,  4.75it/s]
16:27:28 - cmdstanpy - INFO - Chain [1] start processing
16:27:28 - cmdstanpy - INFO - Chain [1] done processing


,y,yhat
0,0.18,0.356748
1,0.33,0.555868
2,0.33,0.337640
3,0.70,1.145565
4,0.88,0.580780
5,0.67,0.539197


##### SARIMAX

In [18]:
model, info_ipca_sarimax, best_value_ipca_sarimax = generate_sarimax_model(train_ipca['valor'], test_ipca['valor'], None, None, n_trials, metrica_erro, tolerancia_exog)

results = model.fit(disp=False)

forecasts_ipca_sarimax = results.forecast(steps=len(test_ipca['valor']))

  0%|          | 0/3 [00:00<?, ?it/s]c:\Users\jhiy2\Documents\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\jhiy2\Documents\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
Best trial: 0. Best value: 1.41693:  33%|███▎      | 1/3 [00:01<00:02,  1.37s/it]c:\Users\jhiy2\Documents\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\jhiy2\Documents\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates

In [19]:
print(f"Valor da métrica: {best_value_ipca_sarimax}")
display(pd.concat([forecasts_ipca_sarimax, test_ipca['valor']], axis=1))

Valor da métrica: 0.3085960597689822


,predicted_mean,valor
2025-11-01,0.084454,0.18
2025-12-01,0.055540,0.33
2026-01-01,0.124103,0.33
2026-02-01,0.122789,0.70
2026-03-01,0.151115,0.88
2026-04-01,0.148051,0.67


#### Selecionar melhor forecast das exógenas

In [20]:
metricas_ipca = np.array([best_value_ipca_prophet, best_value_ipca_sarimax])
metricas_exchange = np.array([best_value_exchange_prophet, best_value_exchange_sarimax])

idx_ipca = np.argmin(metricas_ipca)
idx_exchange = np.argmin(metricas_exchange)

forecast_ipca = pd.Series()
forecast_exchange = pd.Series()
modelo_escolhido_ipca = ''
modelo_escolhido_exchange = ''

if idx_ipca == 0:
    modelo_escolhido_ipca = 'Prophet'
    model_ipca = prophet.Prophet(**info_ipca_prophet)
    model_ipca.fit(pd.concat([df_train_ipca, df_test_ipca]))  
    future = model_ipca.make_future_dataframe(periods=horizonte_previsao, freq='MS')

    forecast_ipca = model_ipca.predict(
        future.tail(horizonte_previsao)
    )

    forecast_ipca.index = forecast_ipca['ds']
    forecast_ipca = forecast_ipca['yhat']
elif idx_ipca == 1:
    modelo_escolhido_ipca = 'SARIMAX'
    seasonal_order = (0, 0, 0, 0)

    if info_ipca_sarimax['seasonal']:
        seasonal_order = (
            info_ipca_sarimax['P'],
            info_ipca_sarimax['D'],
            info_ipca_sarimax['Q'],
            12
        )

    model_ipca = SARIMAX(
        pd.concat([train_ipca['valor'], test_ipca['valor']]),
        order=(
            info_ipca_sarimax['p'],
            info_ipca_sarimax['d'],
            info_ipca_sarimax['q']
        ),
        seasonal_order=seasonal_order,
        trend=info_ipca_sarimax['trend'],
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    results_ipca = model_ipca.fit(disp=False)

    forecast_ipca = results_ipca.forecast(steps=horizonte_previsao)

if idx_exchange == 0:
    modelo_escolhido_exchange = 'Prophet'
    model_exchange = prophet.Prophet(**info_exchange_prophet)
    model_exchange.fit(pd.concat([df_train_exchange, df_test_exchange]))  
    future = model_exchange.make_future_dataframe(periods=horizonte_previsao, freq='MS')

    forecast_exchange = model_exchange.predict(
        future.tail(horizonte_previsao)
    )

    forecast_exchange.index = forecast_exchange['ds']
    forecast_exchange = forecast_exchange['yhat']
elif idx_exchange == 1:
    modelo_escolhido_exchange = 'SARIMAX'
    seasonal_order = (0, 0, 0, 0)

    if info_exchange_sarimax['seasonal']:
        seasonal_order = (
            info_exchange_sarimax['P'],
            info_exchange_sarimax['D'],
            info_exchange_sarimax['Q'],
            12
        )

    model_exchange = SARIMAX(
        pd.concat([train_ex['exchange_rate'], test_ex['exchange_rate']]),
        order=(
            info_exchange_sarimax['p'],
            info_exchange_sarimax['d'],
            info_exchange_sarimax['q']
        ),
        seasonal_order=seasonal_order,
        trend=info_exchange_sarimax['trend'],
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    results_exchange = model_exchange.fit(disp=False)

    forecast_exchange = results_exchange.forecast(steps=horizonte_previsao)

forecast_exchange = forecast_exchange.rename("exchange_rate")
forecast_ipca = forecast_ipca.rename("valor")

exog_previsao = pd.concat([forecast_ipca, forecast_exchange], axis=1)

print(f"Modelo escolhido IPCA: {modelo_escolhido_ipca}")
print(f"Modelo escolhido taxa de câmbio: {modelo_escolhido_exchange}")

16:27:31 - cmdstanpy - INFO - Chain [1] start processing
16:27:31 - cmdstanpy - INFO - Chain [1] done processing
c:\Users\jhiy2\Documents\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\jhiy2\Documents\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


Modelo escolhido IPCA: Prophet
Modelo escolhido taxa de câmbio: SARIMAX


In [21]:
display(forecast_exchange)

2026-05-01    5.018966
2026-06-01    4.999600
2026-07-01    5.014314
Freq: MS, Name: exchange_rate, dtype: float64

In [22]:
display(forecast_ipca)

ds
2026-05-01    0.277025
2026-06-01    0.085463
2026-07-01    0.227956
Name: valor, dtype: float64

#### Modelo da FIPE

##### Separar os dados em treino e teste para fipe

In [23]:
data_ref = pd.to_datetime('2026-04-01')
start = data_ref + relativedelta(months=-tamanho_teste)

train_ipca = df_ipca[df_ipca.index <= start]

test_ipca = df_ipca[df_ipca.index > start]

train_ex = df_ex[df_ex.index <= start]

test_ex = df_ex[df_ex.index > start]

exog_train = pd.concat([train_ipca['valor'], train_ex['exchange_rate']], axis=1)
exog_test = pd.concat([test_ipca['valor'], test_ex['exchange_rate']], axis=1)


df_prophet = pd.concat([df, df_ipca, df_ex], axis=1, sort=False)
df_prophet = df_prophet[['reference_date', 'date', 'brl_price', 'valor', 'exchange_rate']]
df_prophet = df_prophet.rename(columns={
    'reference_date': 'ds',
    'brl_price': 'y'
})

df_train_prophet = df_prophet[
    df_prophet['ds'] <= start
]
df_test_prophet = df_prophet[
    df_prophet['ds'] > start
]

df_train_prophet = df_train_prophet.reset_index(drop=True)
df_test_prophet = df_test_prophet.reset_index(drop=True)

In [24]:
data_ref = pd.to_datetime('2026-04-01')
start = data_ref - relativedelta(months=tamanho_teste)

previsoes_por_sku = {}
modelo_vencedor_por_sku = {}

for sku in skus_teste:
    df_sku_atual = df_previsao.query('sku == @sku')

    train = df_sku_atual[
        df_sku_atual['reference_date'] <= start
    ]
    test = df_sku_atual[
        df_sku_atual['reference_date'] > start
    ]

    train.index = train['reference_date']
    test.index = test['reference_date']

    train = train[train['reference_date'] >= f'{data_ref.year - 5}-01-01']

    modelo_ets, best_value_ets, forecast_ets = criar_ets_fipe_real(train, test, n_trials, metrica_erro, tolerancia_fipe, horizonte_previsao)

    exog_train = exog_train[exog_train.index.isin(train['reference_date'])]

    modelo_sarimax, best_value_sarimax, forecast_sarimax = criar_sarimax_fipe_real(train, test, exog_train, exog_test, exog_previsao, n_trials, metrica_erro, tolerancia_fipe, horizonte_previsao)

    df_sku_atual.index = df_sku_atual['reference_date']
    df_prophet = pd.concat([df_sku_atual, pd.concat([exog_train, exog_test])], axis=1, sort=False)
    df_prophet = df_prophet[['reference_date', 'brl_price', 'valor', 'exchange_rate']]
    df_prophet = df_prophet.rename(columns={
        'reference_date': 'ds',
        'brl_price': 'y'
    })

    df_train_prophet = df_prophet[
        df_prophet['ds'] <= start
    ]
    df_test_prophet = df_prophet[
        df_prophet['ds'] > start
    ]

    df_train_prophet = df_train_prophet.reset_index(drop=True)
    df_test_prophet = df_test_prophet.reset_index(drop=True)
    modelo_prophet, best_value_prophet, forecast_prophet = criar_prophet_fipe_real(df_train_prophet, df_test_prophet, n_trials, metrica_erro, tolerancia_fipe, horizonte_previsao)

    resultados = {
        'ETS': {
            'erro': best_value_ets,
            'forecast': forecast_ets
        },
        'SARIMAX': {
            'erro': best_value_sarimax,
            'forecast': forecast_sarimax
        },
        'PROPHET': {
            'erro': best_value_prophet,
            'forecast': forecast_prophet
        }
    }

    melhor_modelo = min(
        resultados,
        key=lambda x: resultados[x]['erro']
    )

    previsoes_por_sku[sku] = resultados[melhor_modelo]['forecast']
    modelo_vencedor_por_sku[sku] = melhor_modelo

  0%|          | 0/3 [00:00<?, ?it/s]c:\Users\jhiy2\Documents\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
Best trial: 0. Best value: 31.5594:  33%|███▎      | 1/3 [00:00<00:00, 14.14it/s]

                 simulation  brl_price
reference_date                        
2025-11-01      4656.373823     4682.0
2025-12-01      4588.750939     4599.0
2026-01-01      4619.122122     4644.0
2026-02-01      4682.951401     4690.0
2026-03-01      4829.779031     4736.0
2026-04-01      4868.530541     4719.0



c:\Users\jhiy2\Documents\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\jhiy2\Documents\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


31.559441756746008
2026-05-01    4796.606512
2026-06-01    4749.317875
2026-07-01    4770.925999
Freq: MS, Name: simulation, dtype: float64


  0%|          | 0/3 [00:00<?, ?it/s]c:\Users\jhiy2\Documents\trabalho_icd\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
Best trial: 0. Best value: 59.7039:  33%|███▎      | 1/3 [00:00<00:00,  6.09it/s]
c:\Users\jhiy2\Documents\trabalho_icd\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\jhiy2\Documents\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\jhiy2\Documents\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred freque

...

c:\Users\jhiy2\Documents\trabalho_icd\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\jhiy2\Documents\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\jhiy2\Documents\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


c:\Users\jhiy2\Documents\trabalho_icd\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


18.559107783804826
2026-05-01    2472.517205
2026-06-01    2464.348404
2026-07-01    2457.491565
Freq: MS, Name: predicted_mean, dtype: float64


  0%|          | 0/3 [00:00<?, ?it/s]16:29:11 - cmdstanpy - INFO - Chain [1] start processing
16:29:11 - cmdstanpy - INFO - Chain [1] done processing
Best trial: 0. Best value: 41.9898:  33%|███▎      | 1/3 [00:00<00:00,  4.81it/s]
16:29:11 - cmdstanpy - INFO - Chain [1] start processing
16:29:11 - cmdstanpy - INFO - Chain [1] done processing
16:29:11 - cmdstanpy - INFO - Chain [1] start processing
16:29:11 - cmdstanpy - INFO - Chain [1] done processing


41.98979862751706
        y         yhat
0  2511.0  2417.393671
1  2507.0  2411.378066
2  2502.0  2427.625769
3  2496.0  2425.874301
4  2490.0  2476.537173
5  2481.0  2476.667341
ds
2026-05-01    2496.831405
2026-06-01    2492.166437
2026-07-01    2474.805593
Name: yhat, dtype: float64


In [25]:
previsoes_por_sku

{0: 2026-05-01    4796.606512
 2026-06-01    4749.317875
 2026-07-01    4770.925999
 Freq: MS, Name: simulation, dtype: float64,
 1: 2026-05-01    5763.986993
 2026-06-01    5778.959495
 2026-07-01    5799.048677
 Freq: MS, Name: predicted_mean, dtype: float64,
 2: ds
 2026-05-01    5981.550286
 2026-06-01    6032.448624
 2026-07-01    6025.644004
 Name: yhat, dtype: float64,
 3: 2026-05-01    2162.000127
 2026-06-01    2159.003914
 2026-07-01    2156.011853
 Freq: MS, Name: simulation, dtype: float64,
 4: 2026-05-01    2477.922929
 2026-06-01    2474.849085
 2026-07-01    2471.779054
 Freq: MS, Name: simulation, dtype: float64}

In [26]:
modelo_vencedor_por_sku

{0: 'ETS', 1: 'SARIMAX', 2: 'PROPHET', 3: 'ETS', 4: 'ETS'}

## Simulação da solução

A simulação consiste em estimar o lucro que seria obtido pelo cliente se usasse nossa solução para tomada de decisões de compra e venda para um conjunto de carros, para esse exemplo serão considerados 5 carros por um período de 6 meses, onde o modelo prevê um horizonte de um mês e o cliente toma a decisão para o esse mês com base na previsão, depois o modelo é retreinado e é simulado o próximo mês até bater os 6 meses.

### Política:

- Comprar o carro pelo preço atual se a previsão for de subida e for maior em pelo menos 1% do valor atual.
- Vender pelo preço atual se a previsão for de queda e o valor for menor em pelo menos 1% do valor comprado.

### Valores iniciais
* Saldo: R$ 100.000
* Estoque de carros: 0 carros
* Horizonte de previsão: 1 mês

In [28]:
saldo = 100000
horizonte = 1

# Carro = (estoque, preco_compra)

skus = [100, 1832, 2134, 5112, 7023]
carros = {k: [] for k in skus}

start_sim = datetime.datetime(2025, 10, 1)
end_sim = start_sim + relativedelta(months=6)
curr_sim = start_sim

while curr_sim < end_sim:
    # Treinar modelo para exogenas
    ## Separar dados de treino e teste
    train_ipca = df_ipca[df_ipca.index <= curr_sim]
    test_ipca = df_ipca[df_ipca.index > curr_sim]

    train_ex = df_ex[df_ex.index <= curr_sim]
    test_ex = df_ex[df_ex.index > curr_sim]

    exog_train = pd.concat([train_ipca['valor'], train_ex['exchange_rate']], axis=1)
    exog_test = pd.concat([test_ipca['valor'], test_ex['exchange_rate']], axis=1)

    df_prophet = pd.concat([df, df_ipca, df_ex], axis=1, sort=False)
    df_prophet = df_prophet[['reference_date', 'date', 'brl_price', 'valor', 'exchange_rate']]
    df_prophet = df_prophet.rename(columns={
        'reference_date': 'ds',
        'brl_price': 'y'
    })

    df_train_prophet = df_prophet[
        df_prophet['ds'] <= start
    ]
    df_test_prophet = df_prophet[
        df_prophet['ds'] > start
    ]

    df_train_prophet = df_train_prophet.reset_index(drop=True)
    df_test_prophet = df_test_prophet.reset_index(drop=True)


    ## Treinar modelo do cambio
    ### Treinar modelo Sarimax
    model, info_exchange_sarimax, best_value_exchange_sarimax = generate_sarimax_model(train_ex['exchange_rate'], test_ex['exchange_rate'], None, None, n_trials, metrica_erro, tolerancia_exog)
    results = model.fit(disp=False)
    forecasts_ex_sarimax = results.forecast(steps=len(test_ex['exchange_rate']))

    ### Treinar modelo Prophet
    df_train_exchange = df_train_prophet[['ds', 'exchange_rate']].rename(columns={'exchange_rate': 'y'})
    df_test_exchange = df_test_prophet[['ds', 'exchange_rate']].rename(columns={'exchange_rate': 'y'})

    model_exchange, info_exchange_prophet, best_value_exchange_prophet = generate_prophet_model(
        df_train_exchange,
        df_test_exchange,
        [],
        n_trials,
        metrica_erro,
        tolerancia_exog
    )

    model_exchange.fit(
        df_train_exchange,
    )

    forecast_exchange = model_exchange.predict(
        df_test_exchange[['ds']]
    )

    pd.concat([df_test_exchange['y'], forecast_exchange['yhat']], axis=1)

    ## Treinar modelo do IPCA
    ### Treinar modelo Sarimax
    model, info_ipca_sarimax, best_value_ipca_sarimax = generate_sarimax_model(train_ipca['valor'], test_ipca['valor'], None, None, n_trials, metrica_erro, tolerancia_exog)

    results = model.fit(disp=False)

    forecasts_ipca_sarimax = results.forecast(steps=len(test_ipca['valor']))

    ### Treinar modelo Prophet
    df_train_ipca = df_train_prophet[['ds', 'valor']].rename(columns={'valor': 'y'})
    df_test_ipca = df_test_prophet[['ds', 'valor']].rename(columns={'valor': 'y'})

    df_train_prophet = df_train_prophet.ffill()
    df_test_prophet = df_test_prophet.ffill()

    model_ipca, info_ipca_prophet, best_value_ipca_prophet = generate_prophet_model(
        df_train_ipca,
        df_test_ipca,
        [],
        n_trials,
        metrica_erro,
        tolerancia_exog
    )

    model_ipca.fit(
        df_train_ipca
    )

    forecast_ipca = model_ipca.predict(
        df_test_ipca[['ds']]
    )

    pd.concat([df_test_ipca['y'], forecast_ipca['yhat']], axis=1)

    # Selecionar o melhor modelo para cada exogena
    metricas_ipca = np.array([best_value_ipca_prophet, best_value_ipca_sarimax])
    metricas_exchange = np.array([best_value_exchange_prophet, best_value_exchange_sarimax])

    idx_ipca = np.argmin(metricas_ipca)
    idx_exchange = np.argmin(metricas_exchange)

    forecast_ipca = pd.Series()
    forecast_exchange = pd.Series()
    modelo_escolhido_ipca = ''
    modelo_escolhido_exchange = ''

    if idx_ipca == 0:
        modelo_escolhido_ipca = 'Prophet'
        model_ipca = prophet.Prophet(**info_ipca_prophet)
        model_ipca.fit(pd.concat([df_train_ipca, df_test_ipca]))  
        future = model_ipca.make_future_dataframe(periods=horizonte_previsao, freq='MS')

        forecast_ipca = model_ipca.predict(
            future.tail(horizonte_previsao)
        )

        forecast_ipca.index = forecast_ipca['ds']
        forecast_ipca = forecast_ipca['yhat']
    elif idx_ipca == 1:
        modelo_escolhido_ipca = 'SARIMAX'
        seasonal_order = (0, 0, 0, 0)

        if info_ipca_sarimax['seasonal']:
            seasonal_order = (
                info_ipca_sarimax['P'],
                info_ipca_sarimax['D'],
                info_ipca_sarimax['Q'],
                12
            )

        model_ipca = SARIMAX(
            pd.concat([train_ipca['valor'], test_ipca['valor']]),
            order=(
                info_ipca_sarimax['p'],
                info_ipca_sarimax['d'],
                info_ipca_sarimax['q']
            ),
            seasonal_order=seasonal_order,
            trend=info_ipca_sarimax['trend'],
            enforce_stationarity=False,
            enforce_invertibility=False
        )

        results_ipca = model_ipca.fit(disp=False)

        forecast_ipca = results_ipca.forecast(steps=horizonte_previsao)

    if idx_exchange == 0:
        modelo_escolhido_exchange = 'Prophet'
        model_exchange = prophet.Prophet(**info_exchange_prophet)
        model_exchange.fit(pd.concat([df_train_exchange, df_test_exchange]))  
        future = model_exchange.make_future_dataframe(periods=horizonte_previsao, freq='MS')

        forecast_exchange = model_exchange.predict(
            future.tail(horizonte_previsao)
        )

        forecast_exchange.index = forecast_exchange['ds']
        forecast_exchange = forecast_exchange['yhat']
    elif idx_exchange == 1:
        modelo_escolhido_exchange = 'SARIMAX'
        seasonal_order = (0, 0, 0, 0)

        if info_exchange_sarimax['seasonal']:
            seasonal_order = (
                info_exchange_sarimax['P'],
                info_exchange_sarimax['D'],
                info_exchange_sarimax['Q'],
                12
            )

        model_exchange = SARIMAX(
            pd.concat([train_ex['exchange_rate'], test_ex['exchange_rate']]),
            order=(
                info_exchange_sarimax['p'],
                info_exchange_sarimax['d'],
                info_exchange_sarimax['q']
            ),
            seasonal_order=seasonal_order,
            trend=info_exchange_sarimax['trend'],
            enforce_stationarity=False,
            enforce_invertibility=False
        )

        results_exchange = model_exchange.fit(disp=False)

        forecast_exchange = results_exchange.forecast(steps=horizonte_previsao)

    forecast_exchange = forecast_exchange.rename("exchange_rate")
    forecast_ipca = forecast_ipca.rename("valor")

    exog_previsao = pd.concat([forecast_ipca, forecast_exchange], axis=1)

    print(f"Modelo escolhido IPCA: {modelo_escolhido_ipca}")
    print(f"Modelo escolhido taxa de câmbio: {modelo_escolhido_exchange}")

    # Selecionar apenas os dados contendo os skus da simulação com janela deslizante
    df_previsao_list = []
    data_ref = curr_sim

    for sku in skus:
        df = df_fipe.query("sku == @sku").copy()
        df = df.set_index('reference_date')

        meses_totais = relativedelta(data_ref, df.index.min()).years * 12 + relativedelta(data_ref, df.index.min()).months

        if meses_totais < 12:
            print(f"SKU: {sku} não tem dados suficientes ({meses_totais} meses apenas)")
            continue

        meses_esperados = pd.date_range(f'{df.index.min().year}-{df.index.min().month}', f'{data_ref.year}-{data_ref.month}', freq='MS')

        df = df.reindex(meses_esperados)

        df['brand_name'] = df['brand_name'].ffill()
        df['model_name'] = df['model_name'].ffill()
        df['year'] = df['year'].ffill()
        df['fuel_name'] = df['fuel_name'].ffill()

        df['brl_price'] = df['brl_price'].interpolate(method='linear')

        df = df.rename_axis('reference_date').reset_index()

        df.index = df['reference_date']
        df['sku'] = df['sku'].ffill()

        df_previsao_list.append(df)


    # Treinar 3 modelos para cada sku e escolher aquele com melhor desempenho
    df_skus = pd.concat(df_previsao_list, ignore_index=True)
    previsoes_por_sku = {}
    modelo_vencedor_por_sku = {}
    for sku in skus:
        df_sku_atual = df_skus.query('sku == @sku')

        train = df_sku_atual[
            df_sku_atual['reference_date'] <= curr_sim
        ]
        test = df_sku_atual[
            df_sku_atual['reference_date'] > curr_sim
        ]

        train.index = train['reference_date']
        test.index = test['reference_date']

        train = train[train['reference_date'] >= f'{data_ref.year - 5}-01-01']

        modelo_ets, best_value_ets, forecast_ets = generate_ets_model(train, test, n_trials, metrica_erro, tolerancia_fipe)

        exog_train = exog_train[exog_train.index.isin(train['reference_date'])]

        modelo_sarimax, best_value_sarimax, forecast_sarimax = generate_sarimax_model(train, test, exog_train, exog_test, exog_previsao, n_trials, metrica_erro, tolerancia_fipe)

        df_sku_atual.index = df_sku_atual['reference_date']
        df_prophet = pd.concat([df_sku_atual, pd.concat([exog_train, exog_test])], axis=1, sort=False)
        df_prophet = df_prophet[['reference_date', 'brl_price', 'valor', 'exchange_rate']]
        df_prophet = df_prophet.rename(columns={
            'reference_date': 'ds',
            'brl_price': 'y'
        })

        df_train_prophet = df_prophet[
            df_prophet['ds'] <= start
        ]
        df_test_prophet = df_prophet[
            df_prophet['ds'] > start
        ]

        df_train_prophet = df_train_prophet.reset_index(drop=True)
        df_test_prophet = df_test_prophet.reset_index(drop=True)
        modelo_prophet, best_value_prophet, forecast_prophet = generate_prophet_model(df_train_prophet, df_test_prophet, ['valor', 'exchange_rate'], n_trials, metrica_erro, tolerancia_fipe)

        resultados = {
            'ETS': {
                'erro': best_value_ets,
                'forecast': forecast_ets
            },
            'SARIMAX': {
                'erro': best_value_sarimax,
                'forecast': forecast_sarimax
            },
            'PROPHET': {
                'erro': best_value_prophet,
                'forecast': forecast_prophet
            }
        }

        melhor_modelo = min(
            resultados,
            key=lambda x: resultados[x]['erro']
        )

        previsoes_por_sku[sku] = resultados[melhor_modelo]['forecast']
        modelo_vencedor_por_sku[sku] = melhor_modelo

    # Política do cliente
    for sku, carro in carros.items():
        preco_atual = df_skus.query('sku == @sku and reference_date == @curr_sim')
        preco_previsto = previsoes_por_sku[sku].loc(0, 'forecast')

        # Comprar
        if preco_previsto >= preco_atual*1.01 and saldo >= preco_atual:
            print(f'Comprou o sku {sku} por {preco_atual}')
            saldo -= preco_atual
            carros[sku].append((1, preco_atual))
        
        # Vender
        for idx, (estoque, compra) in enumerate(carro):
            if preco_previsto <= compra*0.99:
                print(f'Vendou o sku {sku} por {preco_atual}')
                saldo += preco_atual
                carros[sku].pop(idx)

        
    curr_sim += relativedelta(months=1)

  0%|          | 0/3 [00:00<?, ?it/s]

c:\Users\jhiy2\Documents\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\jhiy2\Documents\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


: 